In [3]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp lặp đơn (Fixed-Point Interation)

Điều kiện hội tụ:

* $(a, b)$ là khoảng cách ly nghiệm của phương trình. 

* $|\varphi'(x)| \leq q < 1 \ \forall x \in [a, b]$ --> đảm bảo nghiệm duy nhất

Ánh xạ co không thoả mãn điều kiện --> tự co lại ánh xạ đến khi thoả mãn

# Áp dụng

Note: Chọn các hàm $\varphi$ ổn định trong tính toán (chọn sao cho độ dốc càng nhỏ càng tốt)

## 1. Tiên nghiệm: 
Đánh giá sai số: $|x_n - x^*| \leq \dfrac{q^n}{1 - q} |x_1 - x_0| < \varepsilon$



### 1.1 Thuật toán
**Input:** Hàm số $f(x)$ và hàm lặp $\varphi(x)$, khoảng cách ly nghiệm $(a,b)$, sai số $\varepsilon$.

**Output:** Nghiệm xấp xỉ $x^*$.

**Thuật toán:**
1. **Khởi tạo:**
   - Nếu $f(a)f(b) > 0 \Rightarrow$ Dừng (khoảng không hợp lệ).
   - Tính $\displaystyle q = \max_{[a,b]} |\varphi'(x)|$. Nếu $q \ge 1 \Rightarrow$ Dừng (không thoả mãn điều kiện hội tụ).
   - Chọn $x_0 \in [a,b]$ tuỳ ý. Tính $x_1 = \varphi(x_0)$.
   - Tính số lần lặp $\displaystyle n = \left\lfloor \log_q \left( \frac{1-q}{|x_1 - x_0|} \varepsilon \right) \right\rfloor + 1$.
   - Đặt vòng lặp $i=0$.
2. **Quá trình lặp:** Khi $i < n$ thực hiện:
   - Tính $x_{i+1} = \varphi(x_i)$.
   - Nếu $f(x_{i+1}) = 0 \Rightarrow x^* = x_{i+1}$, kết thúc thuật toán.
   - Tăng $i \leftarrow i+1$.
3. **Kết thúc:** Trả về nghiệm $x^* = x_n$.

### 1.2 Code
#### 1.2.1 Có cộng thêm sai số làm tròn

In [ ]:
def fixed_point_iteration_v1 (f, phi, dphi, a, b, x_0, q, n, rbl):
    #Initial value
    x = x_0
    results = []
    
    # Fixed-point iteration
    delta = 0 
    diff = abs(phi(x) - x)
    temp_q = q
    for i in range(n):
        # Compute next iteration
        x_new = phi(x)

        delta = temp_q/(1-q) * diff
        temp_q *= q
        
        results.append({
            'i': i,
            'x_(i+1)': x_new,
            'f(x_(i+1))': f(x_new),
            'delta=q^n/(1-q) * |x_1-x_0|': delta
        }) 

        # Prepare for next iteration
        x = x_new
        if (f(x_new) == 0): 
            break

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    if rbl == None:
        print(f"The value of root is: {x}")
    else:
        total_delta = delta + 0.5 * 10**(-rbl)
        print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
        print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x**5 - 17*x + 2
phi = lambda x: (x**5 + 2) / 17
dphi = lambda x: 5*x**4 / 17

a = 0
b = 1
x_0 = a
q = max(np.abs(dphi(x)) for x in [a, b]) # Lipschitz constant (q<1)

n = 15
rbl = 9

fixed_point_iteration_v1 (f, phi, dphi, a, b, x_0, q, n, rbl)

#### Sai số tương đối

Từ công thức sai số tiên nghiệm, suy ra Điều kiện dừng: $n > \log_q \left(\varepsilon \cdot \dfrac{1-q}{|x_1 - x_0|} \right)$

## 2. Hậu nghiệm - Sai số tuyệt đối:

Đánh giá sai số dựa trên công thức: $|x_n - x^*| \leq \dfrac{q}{1 - q} |x_n - x_{n-1}| < \varepsilon$.
### 2.1 Thuật toán
**Input:** Hàm số $f(x)$ và hàm lặp $\varphi(x)$, khoảng cách ly nghiệm $(a,b)$, sai số $\varepsilon$.

**Output:** Nghiệm xấp xỉ $x^*$.

**Thuật toán:**

1. **Khởi tạo:**
   - Nếu $f(a)f(b) > 0 \Rightarrow$ Dừng (khoảng không hợp lệ).
   - Tính $\displaystyle q = \max_{[a,b]} |\varphi'(x)|$. Nếu $q \ge 1 \Rightarrow$ Dừng (không thoả mãn điều kiện hội tụ).
   - Chọn $x_0 \in [a,b]$ tuỳ ý, đặt vòng lặp $i=0$.
2. **Quá trình lặp:**
   - Tính $x_{i+1} = \varphi(x_i)$.
   - Tính sai số ước lượng $\displaystyle \Delta_i = \frac{q}{1-q}|x_{i+1} - x_i|$.
   - Nếu $f(x_{i+1}) = 0$ hoặc $\Delta_i < \varepsilon \Rightarrow x^* = x_{i+1}$, kết thúc thuật toán.
   - Gán $x_i \leftarrow x_{i+1}$, tăng $i \leftarrow i+1$.

### 2.2 Code
#### 2.2.1 Có cộng thêm sai số làm tròn

In [1]:
def fixed_point_iteration_v2 (f, phi, dphi, a, b, x_0, q, n, rbl):
    #Initial value
    x = x_0
    results = []
    
    # Fixed-point iteration
    delta_x = 0
    for i in range(n):
        # Compute next iteration
        x_new = phi(x)
        delta_x = q/(1-q) * abs(x_new - x)
        
        results.append({
            'i': i,
            'x_(i+1)': x_new,
            'f(x_(i+1))': f(x_new),
            'delta_x = q/(1-q) * |x_(i+1) - x_i|': delta_x
        }) 

        # Prepare for next iteration
        x = x_new
        if (f(x_new) == 0): 
            break

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    if rbl == None:
        print(f"The value of root is: {x}")
    else:
        total_delta = delta_x + 0.5 * 10**(-rbl)
        print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
        print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x**5 - 17*x + 2
phi = lambda x: (x**5 + 2) / 17
dphi = lambda x: 5*x**4 / 17

a = 0
b = 1
x_0 = a
q = max(np.abs(dphi(x)) for x in [a, b]) # Lipschitz constant (q<1)

n = 15
rbl = 9

fixed_point_iteration_v2 (f, phi, dphi, a, b, x_0, q, n, rbl)

#### 2.2.2 Không cộng thêm sai số làm tròn

Từ công thức sai số hậu nghiệm, ta có điều kiện dừng: $|x_n - x_{n-1}| < \dfrac{1-q}{q} \varepsilon $

In [7]:
def fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps):
    #Initial value
    x = x_0
    results = []
    # Fixed-point iteration
    i=0
    new_eps = (1-q)/q * eps
    print(f"new_eps = {new_eps}")
    
    while True:
        # Compute next iteration
        x_new = phi(x)
        current_delta_x = abs(x_new - x)
        
        results.append({
            'i': i,
            'x_(i+1)': x_new,
            'f(x_(i+1))': f(x_new),
            'delta_x=|x_(i+1) - x_i|': current_delta_x
        }) 

        x = x_new  
        #stop condition 
        if (f(x_new) == 0): 
            break
        elif current_delta_x < new_eps:
            break
        else:
            i += 1

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    print(f"The value of root with absolute error {eps} is: {x}")

Câu 13: Dùng phương pháp lặp đơn giải gần đúng các phương trình dưới đây với 8 chữ số đáng tin:

a. $x^3+3x^2-1=0$

Nghiệm $x_1$:

Nghiệm $x_2$:

Nghiệm $x_3$:

b. $x^2+4\sin x -1=0$

Nghiệm $x_1$:

Nghiệm $x_2$:

c. $(1.4)^x-x=0$

Nghiệm $x_1$:

Nghiệm $x_2$:

d. $e^x-10x+7=0$

Nghiệm $x_1$:

In [ ]:
f = lambda x: np.exp(x) - 10*x + 7
phi = lambda x: (np.exp(x)+7)/10
dphi = lambda x: (np.exp(x))/10

a = 0.5
b = 1
x_0 = 1
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-8)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

Nghiệm $x_2$:

#### 2.2.3 Tìm nghiệm thoả mãn điều kiện sai số tương đối

Đánh giá sai số dựa trên công thức: $\dfrac{|x_n - x^*|}{|x_n|} \leq \dfrac{q}{1 - q} \cdot \dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta$.  

#### Công thức mục tiêu:

Từ công thức sai số Hậu nghiệm, suy ra Điều kiện dừng: $\dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta \cdot \dfrac{1-q}{q}$

In [10]:
def fixed_point_recursion_relative (f, phi, dphi, a, b, x_0, q, eta):
    #Initial value
    x = x_0 
    results = []
    
    # Fixed-point iteration
    i=0
    new_eta = (1-q)/q * eta
    print(f"new_eta = {new_eta}")
    
    while True:
        # Compute next iteration
        x_new = phi(x)
        current_sigma_x = abs(x_new - x) / abs(x_new)
        
        results.append({
            'i': i,
            'x_(i+1)': x_new,
            'f(x_(i+1))': f(x_new),
            'sigma_x=|x_(i+1)-x_i| / |x_(i+1)|': current_sigma_x
        }) 

        # Chuẩn bị cho lần lặp tiếp theo và để in ra kết quả cuối cùng
        x = x_new

        #stop condition 
        if (f(x) == 0): # x đã được cập nhật thành x_new
            break
        elif current_sigma_x < new_eta:
            break
        else:
            i += 1

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    print(f"The value of root with relative error {eta} is: {x}")

In [ ]:
f = lambda x: x**2 + 3*x - 1
phi = lambda x: 1/(x+3)
dphi = lambda x: -1/(x+3)**2

a = 0
b = 1
x_0 = a
q = max(np.abs(dphi(x)) for x in [a, b]) # Lipschitz constant (q<1)

eta = 5 * 10**(-9)

fixed_point_recursion_relative (f, phi, dphi, a, b, x_0, q, eta)